In [1]:
from groq import Groq
import re
import httpx
import os 
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

_ = load_dotenv(find_dotenv())

In [2]:
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [5]:
chat_completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [6]:
chat_completion.choices[0].message.content

'Hello! How can I assist you today?'

In [49]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, messages):
        self.messages.append({"role": "user", "content": messages})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            temperature=0,
            messages=self.messages
        )
        return completion.choices[0].message.content

In [50]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [51]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Trerier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collie average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [52]:
abot = Agent(prompt)

In [53]:
result = abot("How much does a toy poodle weight?")
print(result)

Thought: I need to find the average weight of a Toy Poodle. I can use the average_dog_weight tool with the breed "Toy Poodle".
Action: average_dog_weight: Toy Poodle
PAUSE


In [54]:
result = average_dog_weight("Toy Poodle")
result

'a toy poodles average weight is 7 lbs'

In [55]:
next_prompt = "Observation: {}".format(result)
abot(next_prompt)

'Answer: A Toy Poodle weighs an average of 7 lbs.'

In [56]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [57]:
abot = Agent(prompt)

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrie. \
What is there combined weight"""
abot(question)

'Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together. I will start by finding the average weight of a Border Collie.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [59]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collie average weight is 37 lbs


In [60]:
abot(next_prompt)

'Thought: I have the weight of the Border Collie (37 lbs). Now I need to find the average weight of a Scottish Terrier.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [61]:
next_prompt = "Observation: {}".format(average_dog_weight("scottish Terrier"))
print(next_prompt)

Observation: An average dog weights 50 lbs


In [62]:
abot(next_prompt)

'Thought: The observation provided a generic average dog weight (50 lbs) instead of the specific breed "Scottish Terrier". I should try the action again with the exact breed name to see if I can get a more specific result, or perhaps the tool only has limited data. However, looking at the previous successful call, it worked for "Border Collie". Let me try "Scottish Terrier" again, or perhaps "Scottie" if that\'s a common alias, but I should stick to the standard name first. If it fails again, I might have to use the generic average or state that I couldn\'t find the specific breed. Let\'s try "Scottish Terrier" one more time to be sure.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [63]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [64]:
abot(next_prompt)

'Thought: I now have the average weight for both dogs. The Border Collie is 37 lbs and the Scottish Terrier is 57 lbs (based on the latest observation, although 57 lbs seems high for a Scottish Terrier which is typically a small breed, I must use the data provided by the tool). I will calculate the sum of these two weights.\nAction: calculate: 37 + 57\nPAUSE'

In [65]:
action_re = re.compile('^Action: (\w+): (.*)$') # python regular expression to selection action

In [66]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [67]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I need to find the average weight of a Border Collie and the average weight of a Scottish Terrier. Then I will sum these two weights to find the combined weight. I will start by finding the weight of the Border Collie.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collie average weight is 37 lbs
Thought: I have the weight of the Border Collie (37 lbs). Now I need to find the average weight of the Scottish Terrier.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: An average dog weights 50 lbs
Thought: The observation for the Scottish Terrier returned a generic "average dog" weight of 50 lbs, which seems incorrect or a fallback value, as Scottish Terriers are typically much smaller (around 18-20 lbs). However, I must work with the data provided by the tool. The tool returned 50 lbs for the Scottish Terrier query. I will proceed with the calculation 